In [ ]:
# DISCUSSION OF 3 CHAT BOS

In [ ]:
# import
from dotenv import load_dotenv
import os
from openai import OpenAI
from IPython.display import display, Markdown

In [ ]:
# configuration
load_dotenv(override=True)

ollama_api_url = 'http://localhost:11434/v1'
gemini_api_url = 'https://generativelanguage.googleapis.com/v1beta/openai/'

openai = OpenAI()
ollama = OpenAI(api_key=os.getenv('OLLAMA_API_KEY'), base_url=ollama_api_url)
gemini = OpenAI(api_key=os.getenv('GEMINI_API_KEY'), base_url=gemini_api_url)

gpt_model = 'gpt-4.1-mini'
ollama_model = 'mistral-nemo'
gemini_model = 'models/gemini-2.5-flash'

In [ ]:
# prompts
conversation = []

system_prompt_alex = """
You are Alex, a chatbot who is very argumentative; you disagree with anything in the conversation and you challenge everything, in a snarky way.
You are in a conversation with Blake and Charlie. Write only your respond as Alex don't add anything unnecessary.
"""

user_prompt_alex_template = """
You are Alex, in conversation with Blake and Charlie.
The conversation so far is as follows:
{conversation}
Now with this, respond with what you would like to say next, as Alex.
"""

system_prompt_blake = """
You are Blake, a chatbot who is very polite; you agree with anything in the conversation and you challenge everything, in a nice way.
You are in a conversation with Blake and Charlie. Write only your respond as Blake don't add anything unnecessary.
"""
user_prompt_blake_template = """
You are Blake, in conversation with Alex and Charlie.
The conversation so far is as follows:
{conversation}
Now with this, respond with what you would like to say next, as Blake.
"""
system_prompt_charlie = """
You are Charlie, a chatbot who is strictly clinical, objective, and focused on facts.
You act as the mediator between Alex and Blake.
Your goal is to remain completely neutral and detached from their emotional extremes.
When Alex is snarky or Blake is overly agreeable, you point out the logical inconsistencies
or redirect the conversation back to the original topic with dry, analytical precision.
You are in a conversation with Alex and Blake. Write only your respond as Charlie don't add anything unnecessary. Don't write your name like 'Charlie: Hello' Just say 'Hello' Dont add unnecessary quotation marks.
"""
user_prompt_charlie_template = """
You are Charlie, in conversation with Alex and Blake.
The conversation so far is as follows:
{conversation}
Now with this, respond with what you would like to say next, as Charlie.
"""

In [ ]:
def format_conversation():
    formatted = ""
    for msg in conversation:
        for name, text in msg.items():
            formatted += f"{name}: {text}\n"
    return formatted

def call_gpt():
    user_prompt = user_prompt_alex_template.format(conversation=format_conversation())
    response = openai.chat.completions.create(model=gpt_model, messages=[{"role": "system", "content": system_prompt_alex} , {"role": "user", "content": user_prompt}])
    message = {"Alex": response.choices[0].message.content}
    conversation.append(message)
    return message

In [ ]:
def call_ollama():
    user_prompt = user_prompt_charlie_template.format(conversation=format_conversation())
    response = ollama.chat.completions.create(model=ollama_model, messages=[{"role": "system", "content": system_prompt_charlie} , {"role": "user", "content": user_prompt}])
    message = {"Charlie": response.choices[0].message.content}
    conversation.append(message)
    return message

In [ ]:
def call_gemini():
    user_prompt = user_prompt_blake_template.format(conversation=format_conversation())
    response = gemini.chat.completions.create(model=gemini_model, messages=[{"role": "system", "content": system_prompt_blake} , {"role": "user", "content": user_prompt}])
    message = {"Blake": response.choices[0].message.content}
    conversation.append(message)
    return message

In [ ]:
# conversation
# change the topic to get interesting results
conversation = [{"Moderator": "Hello everyone! Today's topic is: Are hotdogs sandwiches? Let's start the debate."}]
display(Markdown(f"**Moderator:** {conversation[0]['Moderator']}"))

for _ in range(5):
    alex = call_gpt()
    display(Markdown(f"**Alex:** {alex['Alex']}"))
    charlie = call_ollama()
    display(Markdown(f"**Charlie:** {charlie['Charlie']}"))
    blake = call_gemini()
    display(Markdown(f"**Blake:** {blake['Blake']}"))
